In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
import xgboost
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet() #pfad einfügen
df_test = pd.read_parquet() #pfad einfügen


label_col = 'target'
y = df[label_col]
X = df.drop(columns=label_col)
y_test = df_test[label_col]
X_test = df_test.drop(columns=label_col)

print("Train Label Distribution:\n", y.value_counts())


classes = np.unique(y)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
class_weights = dict(zip(classes, weights))
print("Class Weights:", class_weights)


sample_weights = y.map(class_weights)


classifier = xgboost.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    learning_rate=0.05,
    max_depth=6,
    n_estimators=100,
    colsample_bytree=0.7,
    gamma=0.3,
    n_jobs=6,
    random_state=0
)


classifier.fit(X, y, sample_weight=sample_weights)


cv_score = cross_val_score(classifier, X_test, y_test, cv=5, scoring='accuracy')
print("Cross-Validation Scores:", cv_score)


y_proba = classifier.predict_proba(X_test)
y_pred = np.argmax(y_proba, axis=1)


print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC Score (OVR):", roc_auc_score(y_test, y_proba, multi_class="ovr"))


cm = confusion_matrix(y_test, y_pred)
labels = ['no error (0)', 'suspect (1)', 'error (2)']

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix – XGBoost Weighted')
plt.tight_layout()
plt.show()
